# 🐍 Day 4: SQLAlchemy

- SQLAlchemy ORM basics
- Models and columns
- Queries and filters
- Relationships (one-to-many, many-to-many)

## What is SQLAlchemy?

SQLAlchemy is the most popular Python ORM. Maps Python classes to database tables and lets you query with Python instead of raw SQL.

In [ ]:
# pip install sqlalchemy
from sqlalchemy import create_engine, Column, Integer, String, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime

Base = declarative_base()
engine = create_engine("sqlite:///demo.db", echo=False)
Session = sessionmaker(bind=engine)
session = Session()
print("SQLAlchemy setup with SQLite")

## Defining a Model

Subclass Base, define columns. `__tablename__` sets the table name.

In [ ]:
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, autoincrement=True)
    username = Column(String(50), nullable=False)
    email = Column(String(100), unique=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    def __repr__(self):
        return f"<User(id={self.id}, username={self.username})>"

Base.metadata.create_all(engine)
print("User model and table created")

## Create, Read, Update, Delete

Use session.add(), session.commit(), session.query().

In [ ]:
# Create
user = User(username="alice", email="alice@example.com")
session.add(user)
session.commit()
print("Created:", user.id)

# Read
u = session.query(User).filter_by(username="alice").first()
print("Found:", u)

# Update
u.email = "alice.new@example.com"
session.commit()

# Delete (optional - for demo we keep it)
# session.delete(u)
# session.commit()

## Query Filters

`filter()`, `filter_by()`, `limit()`, `order_by()`, `offset()`

In [ ]:
from sqlalchemy import or_, and_

# All users
all_users = session.query(User).all()

# Filter by column
user = session.query(User).filter(User.username == "alice").first()

# Multiple conditions
users = session.query(User).filter(
    User.username.like("%a%"),
    User.username != "admin"
).limit(10).all()

# Order by
ordered = session.query(User).order_by(User.created_at.desc()).all()

print("Query methods: filter, filter_by, limit, order_by, all, first")

## Relationships

Use `relationship()` and `ForeignKey()` for one-to-many.

In [ ]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class Post(Base):
    __tablename__ = "posts"
    id = Column(Integer, primary_key=True)
    title = Column(String(200))
    user_id = Column(Integer, ForeignKey("users.id"))
    user = relationship("User", backref="posts")

Base.metadata.create_all(engine)
print("Post model with user_id FK and relationship")

In [ ]:
# Create post for user
u = session.query(User).first()
post = Post(title="First post", user_id=u.id)
session.add(post)
session.commit()

# Access via relationship
print("User posts:", u.posts)
print("Post author:", post.user.username)

## Session Rollback

On error, rollback to undo uncommitted changes.

In [ ]:
try:
    session.add(User(username="x"))
    session.commit()
except Exception as e:
    session.rollback()
    print("Rolled back:", e)

## Common Mistakes

- **Forgetting session.commit()**: Changes not persisted
- **Not closing session**: Use session.close() or context manager
- **Circular imports**: Define models before relationships
- **Lazy loading in wrong context**: Use joinedload for relationships when needed
- **Mixing session instances**: Use one session per request in web apps

## Exercises

In [ ]:
# Exercise 1: Create a Product model with id, name, price. Create table and insert one product.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Query all Users, then filter by email containing '@example.com'.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Update a user's username. Use session.query().filter_by().first() then modify and commit.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Add a Category model. Product has category_id FK. Define relationship.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Use session.query(User).count() to get total user count.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Delete a user and commit. What happens to related posts? (depends on FK config)
# YOUR CODE HERE

## Key Takeaways

- Model = class with Column(); Base.metadata.create_all(engine)
- session.add(), commit(), query(Model).filter().first()/all()
- relationship() and ForeignKey for associations
- session.rollback() on error

## 🔜 Next: Day 5 - Flask Auth

Tomorrow: Flask-Login, password hashing, and sessions!